In [5]:
# dowbloading the dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-06-11 08:15:52--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-06-11 08:15:53 (36.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [6]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("chars in dataset: ", len(text))
# let's see the first 66 characters
print(text[:66])

chars in dataset:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:


In [7]:
# let's extract unique chars
chars = list(set(text))
voc_size = len(chars)

print(voc_size, chars)

65 ['i', 'p', 'N', 'd', 'u', 'o', '\n', 'W', '.', '!', 'V', 'v', 'C', 'a', 'm', '&', '3', 'w', ',', 'I', 'M', 'L', 'K', 'B', ' ', 'y', 'E', '$', 'f', 'R', 'l', 'J', ';', 'Q', ':', 'k', 'c', '-', 'X', 'g', 'G', 'x', 'F', 'q', 'e', 'r', '?', 'z', 'Y', 'S', 'P', 'D', 'U', 't', 'h', 'O', 'j', 'n', 'T', "'", 'b', 'H', 'Z', 'A', 's']


In [8]:
# mapping from char to int
ctoi = {ch:i for i,ch in enumerate(chars)}
itoc = {i:ch for i,ch in enumerate(chars)}

# encode / decode strings
encode = lambda s: [ctoi[c] for c in s]
decode = lambda l: ''.join([itoc[i] for i in l])

print(encode("zwix"))
print(decode(encode("zwix")))

[47, 17, 0, 41]
zwix


In [9]:
# let's use torch.tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:66])

torch.Size([1115394]) torch.int64
tensor([42,  0, 45, 64, 53, 24, 12,  0, 53,  0, 47, 44, 57, 34,  6, 23, 44, 28,
         5, 45, 44, 24, 17, 44, 24,  1, 45,  5, 36, 44, 44,  3, 24, 13, 57, 25,
        24, 28,  4, 45, 53, 54, 44, 45, 18, 24, 54, 44, 13, 45, 24, 14, 44, 24,
        64,  1, 44, 13, 35,  8,  6,  6, 63, 30, 30, 34])


In [10]:
# spliting into train and test data
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [11]:
block_size = 8
x = data[:block_size]
y = data[1:block_size + 1]
for i in range(block_size):
    context = x[:i + 1]
    target = y[i]
    print(f"input : {context} | target : {target}")

input : tensor([42]) | target : 0
input : tensor([42,  0]) | target : 45
input : tensor([42,  0, 45]) | target : 64
input : tensor([42,  0, 45, 64]) | target : 53
input : tensor([42,  0, 45, 64, 53]) | target : 24
input : tensor([42,  0, 45, 64, 53, 24]) | target : 12
input : tensor([42,  0, 45, 64, 53, 24, 12]) | target : 0
input : tensor([42,  0, 45, 64, 53, 24, 12,  0]) | target : 53


In [12]:
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[12, 55, 20, 19,  2, 19, 52, 49],
        [53, 44, 44, 30, 24, 39, 45,  5],
        [25,  5,  4, 18, 24, 17, 54,  5],
        [24, 53, 54, 44, 14,  6, 26, 11]])
targets:
torch.Size([4, 8])
tensor([[55, 20, 19,  2, 19, 52, 49, 34],
        [44, 44, 30, 24, 39, 45,  5, 17],
        [ 5,  4, 18, 24, 17, 54,  5, 24],
        [53, 54, 44, 14,  6, 26, 11, 44]])


In [16]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(66)

class BigramLangModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # lookup table to get embedding
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # embedding for each of element of idx
        logits = self.token_embedding_table(idx) # (B, T, C)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C) # (B * T, C)
            targets = targets.view(B * T) # (B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # get predictions
            logits, loss = self(idx)
            # we need the embedding of the last step only
            logits = logits[:, -1, :] # (B, C)
            # we convert the predictions into probabilties
            probs = F.softmax(logits, dim=-1) # (B, C)
            # we roll a dies based on probabities to get next token
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # we add the token we got to idx
            idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1)
        return idx

m = BigramLangModel(voc_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens = 66)[0].tolist()))

torch.Size([32, 65])
tensor(4.9030, grad_fn=<NllLossBackward0>)
iOamkrz&JEL!m
LpfpLBr&c:OmXzCaI:kjHX'qJgUIt.oNR?yLWrlr3Ug-
cKA?uZ?x


In [20]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss)
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens = 66)[0].tolist()))

tensor(2.3702, grad_fn=<NllLossBackward0>)
iniretit,
T: thigre
Ald, d two I's qurpewevon t,
INGH:
IORDUSornr s
